![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 2 — Ingeniería de Datos
**Rol:** CRB_DATA_ANALYTICS | **Tiempo:** 15 min | **Criterio:** Pipeline declarativo, calidad, observabilidad, CI/CD, linaje

> **Tip:** Ve a **Databases > Explorer**, busca el objeto y abre la pestaña **Lineage** para ver la gráfica visual de linaje.

In [ ]:
USE ROLE CRB_DATA_ANALYTICS;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Lanza este prompt en CoCo AHORA
Copia el bloque de abajo y pégalo en Cortex Code **antes de continuar**. CoCo ejecuta todo en paralelo mientras tú avanzas con el resto del notebook.

In [ ]:
-- PROMPT PARA COCO: copiar TODO este bloque y pegar en Cortex Code
--
-- Necesito que hagas estas dos tareas en paralelo. Paraleliza todo lo que puedas.
--
-- TAREA 1: Conectar a la API de la Superfinanciera de Colombia
--   Crea Network Rule + External Access Integration para datos.gov.co
--   Crea una UDF Python que consulte https://www.datos.gov.co/resource/mcec-87by.json
--   Crea la tabla CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA y carga 365 dias de TRM
--
-- TAREA 2: Usa Openflow para crear un conector de ingesta
--   Usa tu skill de Openflow
--   Crea un conector Openflow en esta cuenta para ingestar datos desde una fuente externa
--   Muestra los conectores disponibles y el estado del conector creado
--
-- Al terminar muestra los resultados de ambas tareas.
-- No pidas confirmacion, ejecuta todo de una vez.

---
**Mientras CoCo trabaja, continúa ejecutando los bloques de abajo.**

---

## Bloque 1 — Evidencia: Pipeline declarativo sin Spark

In [ ]:
-- Dynamic Tables: pipeline bronce→plata→oro SIN orquestador
SHOW DYNAMIC TABLES IN DATABASE CREDIBANCO_HOL;

Verificamos que la capa **Silver** (datos limpios y enriquecidos) se genera automáticamente desde la capa Bronze. Sin Jobs de Spark, sin orquestador.

In [ ]:
-- Ver datos transformados en la capa Silver
SELECT COUNT(*) AS filas_silver FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER;
SELECT * FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER LIMIT 5;

El linaje nativo traza la cadena completa de transformación: desde la tabla fuente hasta las capas Silver y Gold — sin metadata externa.

> **Tip:** Ve a **Databases > Explorer**, busca el objeto y abre la pestaña **Lineage** para ver la gráfica visual de linaje.

In [ ]:
-- Linaje nativo: trazar la cadena end-to-end
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER', 'table', 'upstream', 3
));

## Bloque 2 — Ejecutar: Crear nueva DT + ver propagación

Las Dynamic Tables mantienen agregados actualizados automáticamente. Cada 5 minutos, Snowflake recalcula sin necesidad de Tasks manuales ni orquestadores.

In [ ]:
-- Crear una Dynamic Table nueva: agregados por hora
CREATE OR REPLACE DYNAMIC TABLE CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = CREDIBANCO_HOL_WH
AS
SELECT DATE_TRUNC('hour', FECHA_HORA) AS hora,
       CIUDAD, COUNT(*) AS num_tx, SUM(MONTO) AS monto_total
FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
GROUP BY 1, 2;

Verificamos que la Dynamic Table se refresca automáticamente. El `REFRESH_STATE` muestra si está al día o procesando cambios.

In [ ]:
-- Verificar refresh automático
SHOW DYNAMIC TABLES LIKE 'DT_HOURLY_USER%' IN SCHEMA CREDIBANCO_HOL.PAGOS;

Contamos las filas **antes** de insertar. Este número debe aumentar después del refresh de la Dynamic Table.

In [ ]:
-- Conteo ANTES del insert
SELECT 'ANTES' AS momento, COUNT(*) AS filas FROM CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER;

Insertamos un registro nuevo en la tabla fuente. La Dynamic Table aún no lo refleja — necesita un refresh.

In [ ]:
-- Insertar dato nuevo en la tabla fuente
INSERT INTO CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
SELECT 999999, 1, 1, '4111111111111111', '5411', 'Bogota',
       CURRENT_TIMESTAMP(), 9999999, '00', 'ECOMMERCE';

Forzamos el refresh de la Dynamic Table para ver el cambio inmediatamente (en producción esto es automático cada 5 minutos).

In [ ]:
-- Refrescar la DT manualmente para ver el cambio
ALTER DYNAMIC TABLE CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER REFRESH;

Contamos las filas **después** del refresh. El número debe ser mayor — la Dynamic Table capturó el nuevo registro automáticamente.

In [ ]:
-- Conteo DESPUÉS del refresh — debe haber aumentado
SELECT 'DESPUES' AS momento, COUNT(*) AS filas FROM CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER;

## Bloque 2A — Iceberg Tables: Interoperabilidad Abierta
**Apache Iceberg** es el formato abierto que permite que los datos en Snowflake sean accesibles desde cualquier motor (Spark, Trino, Presto). Snowflake soporta Iceberg Tables nativamente — los datos viven en tu storage y Snowflake los gestiona.

In [ ]:
-- Crear una Iceberg Table con datos de autorizaciones aprobadas
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE ICEBERG TABLE CREDIBANCO_HOL.PAGOS.ICE_AUTORIZACIONES_APROBADAS
  CATALOG = 'SNOWFLAKE'
AS
SELECT AUTORIZACION_ID, COMERCIO_ID, CIUDAD, MCC, MONTO, CANAL,
       FECHA_HORA::TIMESTAMP_NTZ(6) AS FECHA_HORA
FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
WHERE CODIGO_RESPUESTA = '00'
LIMIT 1000;

In [ ]:
-- Verificar: Iceberg Tables en la base de datos
SHOW ICEBERG TABLES IN DATABASE CREDIBANCO_HOL;

La tabla Iceberg es accesible desde Snowflake Y desde cualquier motor que soporte Apache Iceberg — **sin copiar datos**. Esto es interoperabilidad real con formato abierto.

## Bloque 2B — dbt Projects: Transformaciones Gobernadas
**dbt (data build tool)** permite definir transformaciones SQL como **modelos versionados** con tests automáticos, documentación y lineage. En Snowflake, dbt corre nativamente como un proyecto desplegado — sin infraestructura adicional.

In [ ]:
-- Verificar que dbt Projects está disponible en esta cuenta
SHOW DBT PROJECTS IN ACCOUNT;

Copia este prompt en **Cortex Code** para crear un proyecto dbt completo:

> **Crea un proyecto dbt en Snowflake con: (1) un modelo llamado stg_autorizaciones_aprobadas que filtre autorizaciones con CODIGO_RESPUESTA='00' desde CREDIBANCO_HOL.PAGOS.AUTORIZACIONES, (2) un modelo mart_comercios_riesgo que agregue por comercio: total transacciones, monto promedio, tasa de rechazo, (3) un test de unique en AUTORIZACION_ID del modelo staging, (4) un test de not_null en COMERCIO_ID del mart. Ejecuta el proyecto y muéstrame los resultados con dbt build.**

Con dbt, cada transformación es:
- **Versionada** en Git
- **Testeada** automáticamente
- **Documentada** con lineage visual
- **Ejecutable** con un solo comando

## Bloque 3 — Verificar lo que CoCo construyó
CoCo debió crear la integración con la API de la Superfinanciera y el flujo Openflow. Ahora verificamos que todo quedó funcionando.

In [ ]:
-- Verificar: Network Rule y External Access Integration creados por CoCo
USE ROLE ACCOUNTADMIN;
SHOW NETWORK RULES IN SCHEMA CREDIBANCO_HOL.PLATAFORMA;
SHOW EXTERNAL ACCESS INTEGRATIONS LIKE '%DATOS%';

Verificamos que la UDF de la TRM funciona — CoCo la creó conectándose a la API de la Superfinanciera. Consultamos los últimos 5 días de TRM **en vivo**.

In [ ]:
-- Verificar: UDF de TRM creada por CoCo — datos REALES de la Superfinanciera
SELECT * FROM TABLE(CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(5))
ORDER BY fecha DESC;

Verificamos la tabla histórica de TRM. CoCo debió cargar 365 días y programar un Task para actualización diaria.

In [ ]:
-- Verificar: tabla TRM_HISTORICA cargada por CoCo
SELECT COUNT(*) AS dias, MIN(fecha) AS desde, MAX(fecha) AS hasta,
       ROUND(AVG(trm), 2) AS trm_promedio
FROM CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA;

Verificamos el flujo Openflow. CoCo debió configurar la replicación desde Postgres hacia Snowflake.

In [ ]:
-- Verificar: flujo Openflow creado por CoCo
SHOW OPENFLOWS IN ACCOUNT;

## Bloque 4 — CoCo
Si quieres que CoCo además programe la actualización diaria de la TRM, copia este prompt:

> **Crea un Task llamado TASK_TRM_DIARIA que ejecute GET_TRM_HISTORICA todos los días a las 8AM Colombia y haga MERGE INTO TRM_HISTORICA para agregar solo los días nuevos. Usa CRON '0 13 * * *' UTC.**

In [ ]:
-- Verificación final
USE ROLE CRB_DATA_ANALYTICS;
SELECT 'T2_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER) AS filas_silver,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER) AS filas_hourly;